# Day 16: Multi-Head Attention

**Goal:** Run multiple attention computations in parallel — each can learn a different pattern. This is the actual block used in every transformer.

### Why "multi-head"?

A single attention head learns ONE attention pattern. But a sentence has many kinds of relationships at once:

```
"The cat that I saw yesterday sat on the mat"

Head 1 might learn:   subject-verb attention ("cat" ↔ "sat")
Head 2 might learn:   adjacent-word attention (bigram-like)
Head 3 might learn:   matching brackets / quotes
Head 4 might learn:   long-range references
```

Multi-head = a **committee of specialists** instead of one generalist.

### Plan for today

1. Slow version: a `nn.ModuleList` of single-head attentions (intuitive)
2. Fast version: one big projection + reshape (what's actually used)
3. Verify they produce identical outputs
4. Train multi-head on Shakespeare, beat Day 15's single-head
5. Visualize each head's attention pattern — see specialization
6. Compare param counts and speed

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

torch.manual_seed(42)

## 1. The Slow (but Clear) Version — a List of Single Heads

Easiest way to understand multi-head attention: make `num_heads` separate single-head attention modules and run each.

In [ ]:
# Single-head causal attention (from Day 15)

class SingleHeadAttention(nn.Module):
    def __init__(self, embed_dim, head_dim, max_seq_len=64):
        super().__init__()
        self.W_q = nn.Linear(embed_dim, head_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, head_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, head_dim, bias=False)
        self.head_dim = head_dim
        self.register_buffer('mask', torch.tril(torch.ones(max_seq_len, max_seq_len)))
    
    def forward(self, x):
        B, T, _ = x.shape
        Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)
        scores = Q @ K.transpose(-2, -1) / (self.head_dim ** 0.5)
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        return weights @ V, weights


# SLOW MULTI-HEAD: list of single heads, run each, concatenate

class MultiHeadAttentionSlow(nn.Module):
    def __init__(self, embed_dim, num_heads, max_seq_len=64):
        super().__init__()
        assert embed_dim % num_heads == 0
        head_dim = embed_dim // num_heads
        
        # One single-head module per head
        self.heads = nn.ModuleList([
            SingleHeadAttention(embed_dim, head_dim, max_seq_len)
            for _ in range(num_heads)
        ])
        # Final projection that mixes the heads
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=False)
    
    def forward(self, x):
        # Run each head, collect outputs
        head_outputs = []
        head_weights = []
        for head in self.heads:
            out, w = head(x)
            head_outputs.append(out)
            head_weights.append(w)
        
        # Concatenate along the last (feature) dim
        combined = torch.cat(head_outputs, dim=-1)   # (B, T, embed_dim)
        return self.W_o(combined), head_weights

# Test
torch.manual_seed(0)
mha_slow = MultiHeadAttentionSlow(embed_dim=16, num_heads=4, max_seq_len=8)
x = torch.randn(2, 8, 16)
out, weights_list = mha_slow(x)

print(f"Input shape:           {x.shape}")
print(f"Output shape:          {out.shape}     (same as input — stackable!)")
print(f"Number of head weights: {len(weights_list)}")
print(f"Each head's weights shape: {weights_list[0].shape}")
print(f"\nParameters: {sum(p.numel() for p in mha_slow.parameters()):,}")

## 2. The Fast (Real) Version — One Big Projection + Reshape

The "list of heads" version does 4 separate matrix multiplies. We can do them all at once with ONE matrix multiply, then reshape.

### The trick

```
W_q has shape (embed_dim, embed_dim).
After applying it:    Q has shape (B, T, embed_dim).

INSTEAD of seeing this as one big projection, INTERPRET it as num_heads
separate projections to head_dim, stacked side-by-side:

  Q (B, T, embed_dim)
    └── view as (B, T, num_heads, head_dim)
    └── transpose to (B, num_heads, T, head_dim)
    └── now each "head" can be processed in parallel
```

The math is IDENTICAL to the slow version. Just packed differently for GPU efficiency.

In [ ]:
class MultiHeadAttention(nn.Module):
    """The standard, fast implementation used in real transformers."""
    
    def __init__(self, embed_dim, num_heads, max_seq_len=64):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.embed_dim = embed_dim
        
        # ONE big projection per role — split into heads later via reshape
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=False)
        
        self.register_buffer('mask', torch.tril(torch.ones(max_seq_len, max_seq_len)))
    
    def forward(self, x):
        B, T, C = x.shape
        H = self.num_heads
        D = self.head_dim
        
        # Project
        Q = self.W_q(x)                # (B, T, C)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Reshape: (B, T, C) → (B, T, H, D) → (B, H, T, D)
        # This is the "splitting into heads" step
        Q = Q.view(B, T, H, D).transpose(1, 2)
        K = K.view(B, T, H, D).transpose(1, 2)
        V = V.view(B, T, H, D).transpose(1, 2)
        # Now Q[b, h] is the queries for batch b, head h
        
        # Attention — works in parallel across all heads
        # (B, H, T, D) @ (B, H, D, T) → (B, H, T, T)
        scores = Q @ K.transpose(-2, -1) / (D ** 0.5)
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)            # (B, H, T, T)
        out = weights @ V                              # (B, H, T, D)
        
        # Recombine heads: (B, H, T, D) → (B, T, H, D) → (B, T, C)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        
        # Final projection that lets heads interact
        return self.W_o(out), weights

# Test the fast version
torch.manual_seed(0)
mha_fast = MultiHeadAttention(embed_dim=16, num_heads=4, max_seq_len=8)
out_fast, weights_fast = mha_fast(x)

print(f"Output shape:  {out_fast.shape}")
print(f"Weights shape: {weights_fast.shape}    (batch, heads, seq, seq)")
print(f"\nParameters: {sum(p.numel() for p in mha_fast.parameters()):,}")
print(f"\nSlow vs fast — same param count: {sum(p.numel() for p in mha_slow.parameters()) == sum(p.numel() for p in mha_fast.parameters())}")

## 3. Speed Comparison: Slow Loop vs Fast Reshape

Same math, very different speeds:

In [ ]:
# Larger inputs to make the speed difference visible

torch.manual_seed(0)
EMBED = 128
NUM_HEADS = 8
SEQ_LEN = 64
BATCH = 16

big_input = torch.randn(BATCH, SEQ_LEN, EMBED)

slow = MultiHeadAttentionSlow(EMBED, NUM_HEADS, max_seq_len=SEQ_LEN)
fast = MultiHeadAttention(EMBED, NUM_HEADS, max_seq_len=SEQ_LEN)

# Warm up
_ = slow(big_input)
_ = fast(big_input)

# Time slow version
N = 50
start = time.time()
for _ in range(N):
    _ = slow(big_input)
slow_time = time.time() - start

start = time.time()
for _ in range(N):
    _ = fast(big_input)
fast_time = time.time() - start

print(f"Configuration: {NUM_HEADS} heads, embed_dim={EMBED}, seq_len={SEQ_LEN}, batch={BATCH}")
print(f"Slow (loop):  {slow_time*1000/N:.2f}ms per call")
print(f"Fast (reshape): {fast_time*1000/N:.2f}ms per call")
print(f"Speedup: {slow_time/fast_time:.1f}x faster")
print(f"\nWith more heads or bigger dims, the gap widens.")
print(f"This is why every real transformer uses the reshape trick.")

## 4. Train a Multi-Head LM on Shakespeare

Same task as Day 14/15: predict the next character. Now using multi-head attention.

In [ ]:
# Shakespeare corpus (same as Day 14/15)

text = """to be or not to be that is the question
whether tis nobler in the mind to suffer
the slings and arrows of outrageous fortune
or to take arms against a sea of troubles
and by opposing end them to die to sleep
no more and by a sleep to say we end
the heart ache and the thousand natural shocks
that flesh is heir to tis a consummation
devoutly to be wished to die to sleep
to sleep perchance to dream ay there's the rub
for in that sleep of death what dreams may come
when we have shuffled off this mortal coil
must give us pause there's the respect
that makes calamity of so long life"""

chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
data = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)

BLOCK_SIZE = 16
BATCH_SIZE = 32
EMBED_DIM = 32
NUM_HEADS = 4

def get_batch():
    starts = torch.randint(0, len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data[s : s + BLOCK_SIZE] for s in starts])
    y = torch.stack([data[s+1 : s+BLOCK_SIZE+1] for s in starts])
    return x, y

print(f"Vocab size: {vocab_size}, embed_dim: {EMBED_DIM}, heads: {NUM_HEADS}")
print(f"Head dim (per head): {EMBED_DIM // NUM_HEADS}")

In [ ]:
# Multi-head attention language model

class MultiHeadLM(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, block_size):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, max_seq_len=block_size)
        self.head = nn.Linear(embed_dim, vocab_size)
    
    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)            # (B, T, embed_dim)
        x, weights = self.attn(x)         # (B, T, embed_dim), (B, H, T, T)
        logits = self.head(x)             # (B, T, vocab_size)
        
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))
        return logits, loss

torch.manual_seed(42)
model = MultiHeadLM(vocab_size, EMBED_DIM, NUM_HEADS, BLOCK_SIZE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Train
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
losses = []
for step in range(2000):
    xb, yb = get_batch()
    _, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 400 == 0:
        print(f"Step {step:4d}: loss={loss.item():.4f}")

print(f"\nFinal loss: {losses[-1]:.4f}")

In [ ]:
# Generate text

def generate(model, start_char='t', max_new=200, temperature=1.0):
    model.eval()
    idx = torch.tensor([[char_to_idx[start_char]]], dtype=torch.long)
    out = [start_char]
    with torch.no_grad():
        for _ in range(max_new):
            idx_crop = idx[:, -BLOCK_SIZE:]
            logits, _ = model(idx_crop)
            last_logits = logits[0, -1, :] / temperature
            probs = F.softmax(last_logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1).item()
            out.append(idx_to_char[next_idx])
            idx = torch.cat([idx, torch.tensor([[next_idx]])], dim=1)
    return ''.join(out)

print("--- Generated text (multi-head model) ---\n")
torch.manual_seed(0)
print(generate(model, start_char='t', max_new=300))

## 5. Visualize Each Head — Do They Specialize?

The most interesting part. Each head learned its OWN attention pattern. Let's see all 4 side by side.

In [ ]:
sample = "to be or not to "  # 16 chars
sample_ids = torch.tensor([[char_to_idx[c] for c in sample]], dtype=torch.long)

model.eval()
with torch.no_grad():
    x = model.tok_emb(sample_ids)
    _, attn_weights = model.attn(x)
    # attn_weights shape: (B=1, num_heads, T, T)

fig, axes = plt.subplots(1, NUM_HEADS, figsize=(20, 5))

for h in range(NUM_HEADS):
    w = attn_weights[0, h].numpy()
    ax = axes[h]
    im = ax.imshow(w, cmap='Blues')
    ax.set_xticks(range(len(sample)))
    ax.set_yticks(range(len(sample)))
    ax.set_xticklabels([repr(c) for c in sample], fontsize=8, rotation=0)
    ax.set_yticklabels([repr(c) for c in sample], fontsize=8)
    ax.set_title(f'Head {h+1}')
    if h == 0:
        ax.set_ylabel('Query (from)')
    ax.set_xlabel('Key (to)')

plt.suptitle(f"Attention patterns of {NUM_HEADS} different heads on '{sample}'", fontsize=13)
plt.tight_layout()
plt.show()

print("Each head's attention pattern is DIFFERENT.")
print("Some focus on the previous char, some on first chars, some look further back.")
print("In a real transformer with 12+ heads, the specialization is much clearer.")

## 6. PyTorch's Built-in `nn.MultiheadAttention`

We built it ourselves to understand it. In practice, you can use PyTorch's built-in version, which is heavily optimized.

The signature is slightly different (it expects a separate Q, K, V input — which lets you do cross-attention too):

In [ ]:
# PyTorch's built-in: nn.MultiheadAttention

builtin_mha = nn.MultiheadAttention(
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    batch_first=True,    # input shape (batch, seq, embed) — matches our convention
    bias=False,
)

x_demo = torch.randn(2, BLOCK_SIZE, EMBED_DIM)

# Build causal mask (1 means "block this connection")
T = BLOCK_SIZE
causal_mask = torch.triu(torch.ones(T, T), diagonal=1).bool()

# Self-attention: same input for Q, K, V
out, weights = builtin_mha(x_demo, x_demo, x_demo, attn_mask=causal_mask)

print(f"Input shape:   {x_demo.shape}")
print(f"Output shape:  {out.shape}     (same as input)")
print(f"Weights shape: {weights.shape}  (averaged across heads by default)")
print(f"\nThis is the production-ready version. Our hand-built version uses the same math.")

---

## Exercises

1. **Effect of head count:** Train with `num_heads=1`, `num_heads=2`, `num_heads=8` (keep `embed_dim=32`). Compare final losses. More heads = better?

2. **Make a single head worth 4:** Set `num_heads=1` but `head_dim=128`. Compare param counts and loss vs `num_heads=4, head_dim=32`. Same params, different structure.

3. **Inspect head similarity:** After training, compute the dot product between each pair of heads' final weights. Are some heads redundant?

4. **Prune useless heads:** Try zeroing out the output of one head before generation. Does quality drop a lot, a little, or not at all?

5. **Compare to PyTorch's built-in:** Initialize your `MultiHeadAttention` and `nn.MultiheadAttention` with identical weights. Verify they produce identical outputs (within float precision).

---

## Key Takeaways

### The recipe

```python
1. Project x to Q, K, V (each shape: B, T, embed_dim)
2. Reshape Q, K, V into heads: (B, T, embed) → (B, H, T, head_dim)
3. Attention per head, IN PARALLEL: scores = Q @ K.T, softmax, @ V
4. Concat heads back: (B, H, T, head_dim) → (B, T, embed_dim)
5. Final output projection W_O
```

### Why multi-head wins

- **Specialization:** Different heads learn different patterns
- **Same compute as one big head:** Parameter count is identical
- **Parallel:** All heads compute simultaneously
- **Standard in every transformer:** GPT, BERT, Llama, Claude

### Where we are

```
Day 13: RNN                                  ✓
Day 14: Bigram LM                            ✓
Day 15: Single-head attention                ✓
Day 16: MULTI-HEAD attention                 ✓ ← YOU ARE HERE
Day 17: Positional encoding                  ← NEXT (telling model about order)
Day 18: Project — attention-based generator
Day 19: TRANSFORMER BLOCK (puts it all together)
Day 20+: Full mini GPT
```

### A key observation

Multi-head attention doesn't yet have any sense of WHERE each token is in the sequence. It just sees a SET of tokens. That's surprising — and it's why we need Day 17's **positional encoding**. Without it, attention is "permutation invariant" and can't distinguish "cat sat dog" from "dog sat cat."

**Tomorrow:** Positional encoding. Adding location information to embeddings so the model knows order.